# 게이트 v2 + 마스킹 + 회색 띠 증강 — ConvNeXt-Tiny, 시드 0·1·2

**지난번(시드 0) 결과:** 정확도 0.972, 오탐 27, 미탐 69. 오탐은 거의 사라졌지만 출력 후반(fold 4)에서 크롭이 출력물로 꽉 찬 언더필 사진을 정상으로 놓쳤다(미탐 14 → 55). 학습 사진 중 꽉 찬(회색 없는) 사진의 75%가 정상이라, 모델이 "회색이 없으면 정상"으로 기운 것으로 보인다.

**이번에 바꾼 것 (`--bed-erase 0.5`):** 학습할 때만 사진의 절반에 마스킹과 같은 회색 띠를 가장자리 1~2곳에 무작위로 칠한다(두께 10~40%). 정상·언더필 구분 없이 칠하니까 회색 면적이 정답의 단서가 되지 못한다. 평가와 실제 판정에는 칠하지 않는다. 나머지(게이트 v2, 마스킹, 고정 크롭, finetune 20 epoch, 5-fold)는 지난번과 같다.

- 시드 0 → 1 → 2 순서, 시드마다 fold 4 → 0 → 3 → 1 → 2 순서(문제였던 fold 4를 먼저 확인). 시드 하나 약 1시간 20분, 전체 약 4시간.
- 시드마다 결과가 따로 저장되고, fold가 끝날 때마다 중간 결과도 저장된다.
- (선택) 7번 셀: 회색 띠 **없이** 시드 1·2도 돌려서 시드 3개끼리 공정하게 비교. 약 2시간 40분 추가.

**Drive `산공종설/`에 필요한 파일**
- `underfill_originals_4000.zip`: 이미 있음
- `pipeline_train_bundle_20261002_mask_erase.zip`: **새로 올릴 것**(학습 스크립트 `--bed-erase` 추가본, 게이트 v2 `filament_gate.py`, 분할표)

노트북은 **파일 > 노트북 업로드**로 연다. 4번 셀을 누르고 **런타임 > 이후 셀 실행**을 하면 학습 → 결과 → CSV 묶기 → 런타임 반납까지 한 번에 한다(선택 셀 7번은 기본으로 건너뜀).

## 1. GPU 확인

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "GPU 런타임이 아님: 런타임 > 런타임 유형 변경 > GPU"

## 2. Drive 연결 + 파일 확인

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
import os
DRIVE_DIR = "/content/drive/MyDrive/산공종설"
RESULTS = f"{DRIVE_DIR}/results/pipeline_fc512_texture_v2_mask_erase"   # 이번 결과
NOERASE = f"{DRIVE_DIR}/results/pipeline_fc512_texture_v2_mask"         # 지난번(회색 띠 없음) 결과
OLD = f"{DRIVE_DIR}/results/pipeline_fc512_gate"                         # 기존 게이트 결과
os.makedirs(RESULTS, exist_ok=True)
for f in ("underfill_originals_4000.zip", "pipeline_train_bundle_20261002_mask_erase.zip"):
    print("OK  " if os.path.exists(f"{DRIVE_DIR}/{f}") else "없음", f)

## 3. 압축 해제
`v2-mask`가 보이는 줄, `--bed-erase`가 보이는 줄, 마지막 줄에 4000이 나와야 한다.

In [ ]:
!cp "{DRIVE_DIR}/underfill_originals_4000.zip" "{DRIVE_DIR}/pipeline_train_bundle_20261002_mask_erase.zip" /content/
!unzip -q -o /content/underfill_originals_4000.zip -d /content/orig
!unzip -q -o /content/pipeline_train_bundle_20261002_mask_erase.zip -d /content/code
!grep -m1 VERSION /content/code/filament_gate.py
!grep -m1 -o -- "--bed-erase" /content/code/train_pilot_classifier.py
!find /content/orig -name "*.jpg" | wc -l

## 4. 본 실행 (회색 띠 증강, 시드 0 → 1 → 2)
- 시작하고 몇 분 뒤 `held 551 images {'normal': 257, 'underfill_50': 294}`가 나오면 정상이다.
- fold 하나가 끝날 때마다 한 줄씩 나온다(약 15분 간격). 첫 줄이 fold 4다. 지난번 fold 4는 정확도 0.919, 미탐 55였다.
- **끊겼을 때:** 이 셀을 다시 실행하면 끝난 시드는 건너뛰고, 하던 시드는 처음부터 다시 한다.
- **시간이 모자라면:** 셀을 멈추고(■) 5번부터 실행한다. 끝난 시드·fold까지 보여 준다.

In [ ]:
import shutil
COMMON = ("--folds-csv /content/code/folds.csv --pilot-root /content/orig --input-column source "
          "--fixed-crop 0 250 1000 1250 --img-size 512 --modes finetune --fast "
          "--gate texture --min-filament 0.1 --mask-bed")
HIDE = r"^ \|^{\|^}"     # 학습 끝의 긴 요약(JSON)을 화면에서 숨김. 같은 내용이 CSV에 있음
ARCH, SEEDS, FOLD_ORDER = "convnext_tiny", [0, 1, 2], "4 0 3 1 2"

def run(base, seed, extra):
    out = f"{base}/{ARCH}_seed{seed}_all"
    if os.path.exists(f"{out}/summary.csv"):
        print("이미 끝남:", out)
        return
    shutil.rmtree(out, ignore_errors=True)   # 반만 끝난 폴더 정리
    print(f"\n===== {os.path.basename(base)} seed {seed}")
    !PYTHONUNBUFFERED=1 python /content/code/train_pilot_classifier.py {COMMON} {extra} --arch {ARCH} --seeds {seed} --folds {FOLD_ORDER} --output "{out}" | grep -v '{HIDE}'

for SEED in SEEDS:
    run(RESULTS, SEED, "--bed-erase 0.5")

## 5. 결과
- 위 표: 시드·fold별 정확도와 오탐/미탐. `회색 띠`가 이번, `띠 없음`이 지난번(시드 0만 있음, 7번을 돌리면 1·2도 생김), `기존 게이트`가 맨 처음 결과다.
- 아래 표: 5 fold가 끝난 시드의 합산 점수와 시드 평균.
- 참고(같은 3,449장): 빠른 모델 정확도 0.984, 오탐 32, 미탐 25 / 띠 없음 시드 0: 0.972, 오탐 27, 미탐 69

**이 셀 출력을 복사해서 Claude에게 붙여 주면 된다.**

In [ ]:
import pandas as pd, json, glob
def fold_rows(base, name):
    rows = []
    for p in sorted(glob.glob(f"{base}/{ARCH}_seed*_all/runs/finetune/seed*/fold*/metrics.json")):
        m = json.load(open(p))
        rows.append({"조건": name, "seed": m["seed"], "fold": m["fold"], "n": m["val_n"], "acc": m["val_accuracy"],
                     "FP": m["val_fp"], "FN": m["val_fn"], "auroc": m["val_auroc"]})
    return rows
fr = pd.DataFrame(fold_rows(RESULTS, "회색 띠") + fold_rows(NOERASE, "띠 없음"))
if len(fr):
    print(fr.pivot_table(index=["조건", "seed"], columns="fold", values="acc").round(3).to_string())
    print()
    print(fr.pivot_table(index=["조건", "seed"], columns="fold", values=["FP", "FN"], aggfunc="sum").to_string())
summ = []
for base, name in ((RESULTS, "회색 띠"), (NOERASE, "띠 없음")):
    for p in sorted(glob.glob(f"{base}/{ARCH}_seed*_all/summary.csv")):
        s = pd.read_csv(p, encoding="utf-8-sig").iloc[0]
        v = pd.read_csv(p.replace("summary.csv", "vote_metrics.csv"), encoding="utf-8-sig").iloc[0]
        summ.append({"조건": name, "seed": int(v.seed), "acc": s.pooled_mean_accuracy, "auroc": s.pooled_mean_auroc,
                     "FP": int(s.pooled_fp_sum_over_seeds), "FN": int(s.pooled_fn_sum_over_seeds), "vote_acc": v.vote_accuracy})
if summ:
    sm = pd.DataFrame(summ)
    print("\n[5 fold 합산]\n" + sm.round(3).to_string(index=False))
    g = sm.drop(columns="seed").groupby("조건")
    avg = g.mean().round(3); avg.insert(0, "n_seeds", g.size())
    print("\n[시드 평균]\n" + avg.to_string())

## 6. 결과 CSV 묶기
가중치를 뺀 CSV만 Drive `results/pipeline_fc512_mask_erase_csv.zip`으로 묶는다(이번 결과 + 띠 없음 결과). 이 zip을 받아서 Claude에게 주면 된다.

In [ ]:
import zipfile
ZIP = f"{DRIVE_DIR}/results/pipeline_fc512_mask_erase_csv.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for base in (RESULTS, NOERASE):
        for p in glob.glob(f"{base}/**/*", recursive=True):
            if p.endswith((".csv", ".json", ".txt")):
                z.write(p, os.path.join(os.path.basename(base), os.path.relpath(p, base)))
print(ZIP, f"{os.path.getsize(ZIP) / 2**20:.1f} MiB")

## 7. (선택) 회색 띠 없이 시드 1·2 — 공정 비교용
회색 띠 효과를 시드 3개끼리 비교하려면 띠 없는 조건도 시드 1·2가 필요하다(시드 0은 지난번에 끝남). 약 2시간 40분. 기본은 꺼져 있어서(`RUN_NOERASE = False`) "이후 셀 실행"을 해도 건너뛴다. 돌리려면 `True`로 바꿔 이 셀을 실행하고, 끝나면 5번·6번을 다시 실행한다.

In [ ]:
RUN_NOERASE = False   # True로 바꾸면 실행
if RUN_NOERASE:
    for SEED in [1, 2]:
        run(NOERASE, SEED, "")
else:
    print("건너뜀 (RUN_NOERASE = False)")

## 8. 런타임 반납

In [ ]:
from google.colab import drive, runtime
drive.flush_and_unmount()
runtime.unassign()